# OpenJev: every question type

Runs every example from [docs/examples.md](../docs/examples.md) with `jev.run(request)`, then the shorthand methods and the pipeline.

`OpenJev.load("openjev")` uses `checkpoints/openjev` if you have trained one. Without it, it loads ModernBERT-base with **untrained** scoring heads and warns: the output shapes are right, but the numbers are close to uniform and mean nothing until you train (`python scripts/train.py --config configs/openjev.json`).

In [2]:
import json
import os
import sys

# Run from the repo root so `import openjev` and checkpoints/ resolve.
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebook" else os.getcwd()
os.chdir(ROOT)
sys.path.insert(0, ROOT)

from openjev import OpenJev, OpenJevPipeline

jev = OpenJev.load("openjev")


def show(request):
    """Run a Jev request and print the response."""
    print(json.dumps(jev.run(request), indent=2))

/Users/vignesh/Documents/personal/OpenJev/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/vignesh/Documents/personal/OpenJev/openjev/infer.py:102: UserWarning: No trained OpenJev checkpoint for 'openjev'; the scoring heads are untrained and their probabilities are meaningless. Train first with scripts/train.py.
  warnings.warn(
Loading weights: 100%|██████████| 134/134 [00:00<00:00, 14064.98it/s]
[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## `noul`: true or false

`criteria` is optional; when given, it describes what `true` and `false` mean. The answer is P(yes).

In [3]:
show({
    "state": "Help! My payouts have been failing for 3 days.",
    "model": "jev-latest",
    "questions": {
        "is_urgent": {
            "type": "noul",
            "instructions": "Does this message convey urgency?",
            "criteria": {
                "true": "Explicitly needs immediate attention",
                "false": "No urgency expressed",
            },
        }
    },
})

{
  "model": "openjev",
  "answers": {
    "is_urgent": {
      "type": "noul",
      "noul": 0.4837438464164734
    }
  },
  "usage": {
    "input_tokens": 57,
    "output_tokens": 0
  },
  "elapsed": 134
}


## `choice`: pick exactly one

`criteria` maps each option to a description of when it applies. Probabilities sum to 1.

In [4]:
show({
    "state": "Help! My payouts have been failing for 3 days.",
    "model": "jev-latest",
    "questions": {
        "department": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {
                "billing": "Payments, invoicing, refunds",
                "technical": "Bugs, outages, integrations",
                "sales": "Pricing, upgrades, new accounts",
            },
        }
    },
})

{
  "model": "openjev",
  "answers": {
    "department": {
      "type": "choice",
      "choice": "technical",
      "probabilities": {
        "billing": 0.31521958112716675,
        "technical": 0.35523244738578796,
        "sales": 0.32954809069633484
      },
      "confidence": 0.0011168552942808496
    }
  },
  "usage": {
    "input_tokens": 97,
    "output_tokens": 0
  },
  "elapsed": 96
}


## `score`: a point on an ordered scale

`criteria` lists 2 to 10 levels, low to high. `legend` and `probabilities` are keyed by level index from `"0"`, and `score` is the expected index, so it can fall between levels.

In [5]:
show({
    "state": "Help! My payouts have been failing for 3 days.",
    "model": "jev-latest",
    "questions": {
        "frustration": {
            "type": "score",
            "instructions": "How frustrated is the customer?",
            "criteria": ["Calm", "Frustrated", "Very angry"],
        }
    },
})

{
  "model": "openjev",
  "answers": {
    "frustration": {
      "type": "score",
      "score": 1.0023301839828491,
      "legend": {
        "0": "Calm",
        "1": "Frustrated",
        "2": "Very angry"
      },
      "probabilities": {
        "0": 0.3222311735153198,
        "1": 0.35320770740509033,
        "2": 0.3245612382888794
      },
      "confidence": 0.0008050729308222149
    }
  },
  "usage": {
    "input_tokens": 88,
    "output_tokens": 0
  },
  "elapsed": 84
}


## All three in one request

Every question about one state goes through the encoder in one batch.

In [ ]:
show({
    "state": "Help! My payouts have been failing for 3 days.",
    "model": "jev-latest",
    "questions": {
        "urgent": {
            "type": "noul",
            "instructions": "Does this message convey urgency?",
            "criteria": {"true": "Explicitly needs immediate attention", "false": "No urgency expressed"},
        },
        "department": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {
                "billing": "Payments, invoicing, refunds",
                "technical": "Bugs, outages, integrations",
                "sales": "Pricing, upgrades, new accounts",
            },
        },
        "frustration": {
            "type": "score",
            "instructions": "How frustrated is the customer?",
            "criteria": ["Calm", "Frustrated", "Very angry"],
        },
    },
})

## `noul` without `criteria`

In [ ]:
show({
    "state": "I was charged twice for my March invoice.",
    "model": "jev-latest",
    "questions": {
        "mentions_refund": {"type": "noul", "instructions": "Is the customer asking for money back?"}
    },
})

## Structured `state`

`state` can be an object or array; OpenJev encodes it as JSON.

In [ ]:
show({
    "state": {
        "subject": "Cannot log in after password reset",
        "body": "I reset my password an hour ago and the new one is rejected. I have a demo with a client at 3pm.",
        "plan": "enterprise",
        "previous_tickets": 4,
    },
    "model": "jev-latest",
    "questions": {
        "escalate": {
            "type": "noul",
            "instructions": "Should this ticket skip the queue?",
            "criteria": {
                "true": "Enterprise customer blocked with a deadline today",
                "false": "Can wait for normal handling",
            },
        }
    },
})

## Structured `instructions`

Put the question and the data it needs in one object, and refer to the data by field name.

In [ ]:
show({
    "state": "Hi, the blender I ordered 40 days ago stopped working. Can I return it?",
    "model": "jev-latest",
    "questions": {
        "eligible": {
            "type": "noul",
            "instructions": {
                "question": "Is this return allowed under the policy?",
                "policy": "Items can be returned within 30 days of delivery. Defective items can be returned within 1 year.",
            },
            "criteria": {"true": "The request fits the policy", "false": "The request is outside the policy"},
        }
    },
})

## `choice` with more options (up to 255)

In [ ]:
show({
    "state": "The app crashes every time I open the camera on my Pixel 8.",
    "model": "jev-latest",
    "questions": {
        "component": {
            "type": "choice",
            "instructions": "Which part of the product is affected?",
            "criteria": {
                "auth": "Login, signup, passwords, two-factor codes",
                "payments": "Cards, charges, payouts, invoices",
                "media": "Camera, photo upload, video playback",
                "notifications": "Push notifications, emails, SMS",
                "sync": "Data missing or out of date across devices",
                "other": "None of the above",
            },
        }
    },
})

## `score` with object levels

Levels can be objects; the `legend` returns each one as given.

In [ ]:
show({
    "state": "Checkout has been down for all EU customers for 20 minutes.",
    "model": "jev-latest",
    "questions": {
        "severity": {
            "type": "score",
            "instructions": "How severe is this incident?",
            "criteria": [
                {"level": "SEV4", "description": "Cosmetic issue, no user impact"},
                {"level": "SEV3", "description": "Minor feature broken, workaround exists"},
                {"level": "SEV2", "description": "Major feature broken for some users"},
                {"level": "SEV1", "description": "Core flow down for many users"},
            ],
        }
    },
})

## Several questions over a review

In [ ]:
show({
    "state": "Delivery was quick but the headphones stopped charging after two days. Support never replied.",
    "model": "jev-latest",
    "questions": {
        "satisfaction": {
            "type": "score",
            "instructions": "How satisfied is the customer overall?",
            "criteria": ["Very unhappy", "Unhappy", "Neutral", "Happy", "Very happy"],
        },
        "topic": {
            "type": "choice",
            "instructions": "What is the main complaint about?",
            "criteria": {
                "shipping": "Speed or condition of delivery",
                "quality": "The product broke or does not work",
                "support": "Slow or unhelpful customer service",
                "price": "Cost or value for money",
            },
        },
        "needs_reply": {"type": "noul", "instructions": "Should someone from support contact this customer?"},
    },
})

## `multi`: pick any number

Not part of the Jev API, but OpenJev supports it in requests too. Each option gets an independent probability; `selected` holds those at or above `threshold`.

In [ ]:
print(json.dumps(jev.run({
    "state": "Arrived two weeks late and the box was crushed.",
    "questions": {
        "problems": {
            "type": "multi",
            "instructions": "Which problems does the customer report?",
            "criteria": {"late": "Delivered after the promised date",
                         "damaged": "Item or packaging arrived broken",
                         "wrong item": "Received something they did not order"},
        }
    },
}, threshold=0.5), indent=2))

## Shorthand methods

One question at a time, without building a request.

In [ ]:
state = "Help! My payouts have been failing for 3 days."
print(jev.noul(state, instructions="Does this convey urgency?"))
print(jev.choice(state, {"billing": "Payments, invoicing, refunds", "technical": "Bugs, outages"},
                 instructions="Which team should handle this?"))
print(jev.score(state, ["Calm", "Frustrated", "Very angry"], instructions="How frustrated is the customer?"))
print(jev.multi(state, ["payouts", "login", "pricing"], instructions="Which topics does it mention?"))

## Pipeline

`OpenJevPipeline` takes the same questions as flat dicts (each with its own `state`) and batches them.

In [ ]:
pipe = OpenJevPipeline(model=jev.model, tokenizer=jev.tokenizer, device=jev.device)
questions = [
    {"type": "score", "state": state, "instructions": "How frustrated is the customer?",
     "criteria": ["Calm", "Frustrated", "Very angry"]},
    {"type": "choice", "state": state, "instructions": "Which team should handle this?",
     "criteria": {"billing": "Payments, invoicing, refunds", "technical": "Bugs, outages"}},
    {"type": "noul", "state": state, "instructions": "Does this convey urgency?"},
]
for result in pipe(questions, batch_size=2):
    print(result)